In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import precision_score, recall_score
from sklearn.model_selection import KFold


In [2]:
# Load the dataset
url = (
    "https://raw.githubusercontent.com/DataTalksClub/"
    "machine-learning-zoomcamp/main/cohorts/2026/data/"
    "course_lead_scoring_2026.csv"
)

df = pd.read_csv(url)

# Inspect the dataset
print("Dataset shape:", df.shape)
print("\nFirst five rows:")
print(df.head())

print("\nMissing values:")
print(df.isnull().sum())

print("\nTarget distribution:")
print(df["converted"].value_counts())


Dataset shape: (5000, 9)

First five rows:
      lead_source       industry employment_status       location  \
0  organic_search     technology          employed         europe   
1    social_media     technology          employed  south_america   
2        referral         retail          employed         europe   
3        paid_ads  manufacturing           student            NaN   
4        referral  manufacturing           student  north_america   

   annual_income  number_of_courses_viewed  interaction_count  lead_score  \
0        88160.0                         3                  4        0.64   
1        72688.0                         3                  7        0.72   
2        44697.0                         3                  4        0.58   
3            NaN                         3                  6        0.57   
4        24062.0                         4                  5        0.62   

   converted  
0          1  
1          1  
2          1  
3          0  
4   

## Question 1: Split the dataset into training, validation, and test sets

In [3]:
# Separate the target
df["converted"] = df["converted"].astype(int)

# Identify numerical and categorical features
numerical = df.select_dtypes(include=["number"]).columns.tolist()
numerical.remove("converted")

categorical = df.select_dtypes(exclude=["number"]).columns.tolist()

# Fill missing values
df[numerical] = df[numerical].fillna(0.0)
df[categorical] = df[categorical].fillna("NA")

# Split the data exactly as specified
df_full_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=1
)

df_train, df_val = train_test_split(
    df_full_train,
    test_size=0.25,
    random_state=1
)

# Separate the training target
y_train = df_train["converted"].values

print("\nFull training set:", df_full_train.shape)
print("Training set:", df_train.shape)
print("Validation set:", df_val.shape)
print("Test set:", df_test.shape)

# Candidate numerical features
features = [
    "lead_score",
    "number_of_courses_viewed",
    "interaction_count",
    "annual_income",
]

# Calculate AUC for each feature
results = {}

for feature in features:
    auc = roc_auc_score(y_train, df_train[feature])

    # Reverse the direction if AUC is below 0.5
    if auc < 0.5:
        auc = roc_auc_score(y_train, -df_train[feature])

    results[feature] = auc

# Sort features by AUC
for feature, auc in sorted(
    results.items(),
    key=lambda item: item[1],
    reverse=True
):
    print(f"{feature}: {auc:.6f}")

print("\nFeature with highest AUC:", max(results, key=results.get))


Full training set: (4000, 9)
Training set: (3000, 9)
Validation set: (1000, 9)
Test set: (1000, 9)
lead_score: 0.788225
interaction_count: 0.764920
number_of_courses_viewed: 0.722989
annual_income: 0.608198

Feature with highest AUC: lead_score


## Question 2: Train a logistic regression model using the best feature

In [4]:
# Separate target variables
y_train = df_train["converted"].values
y_val = df_val["converted"].values

# Remove the target from the input features
df_train_features = df_train.drop(columns=["converted"])
df_val_features = df_val.drop(columns=["converted"])

# Convert dataframes into dictionaries
train_dicts = df_train_features.to_dict(orient="records")
val_dicts = df_val_features.to_dict(orient="records")

# Apply one-hot encoding
dv = DictVectorizer(sparse=True)

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

print("Training matrix:", X_train.shape)
print("Validation matrix:", X_val.shape)

# Train the model
model = LogisticRegression(
    solver="liblinear",
    C=1.0,
    max_iter=1000
)

model.fit(X_train, y_train)

# Predict probabilities for the positive class
y_pred = model.predict_proba(X_val)[:, 1]

# Calculate validation AUC
auc = roc_auc_score(y_val, y_pred)

print(f"Validation AUC: {auc:.3f}")


Training matrix: (3000, 28)
Validation matrix: (1000, 28)
Validation AUC: 0.732


## Question 3: Find the threshold where precision and recall are closest

In [5]:
thresholds = np.arange(0.0, 1.01, 0.01)

results = []

for threshold in thresholds:
    y_pred_binary = (y_pred >= threshold).astype(int)

    precision = precision_score(
        y_val, y_pred_binary, zero_division=0
    )

    recall = recall_score(
        y_val, y_pred_binary, zero_division=0
    )

    # Ignore thresholds where both metrics are zero
    if precision == 0 and recall == 0:
        continue

    results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "difference": abs(precision - recall)
    })

results_df = pd.DataFrame(results)

# Find the threshold where precision and recall are closest
best_row = results_df.loc[results_df["difference"].idxmin()]

print(best_row)

threshold     0.630000
precision     0.732759
recall        0.725256
difference    0.007503
Name: 63, dtype: float64


## Question 4: Find the threshold with the maximum F1 score

In [6]:
results = []

for threshold in np.arange(0.0, 1.01, 0.01):
    y_pred_binary = (y_pred >= threshold).astype(int)

    precision = precision_score(
        y_val, y_pred_binary, zero_division=0
    )

    recall = recall_score(
        y_val, y_pred_binary, zero_division=0
    )

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

f1_df = pd.DataFrame(results)

best_f1 = f1_df.loc[f1_df["f1"].idxmax()]

print("\nQuestion 4: Maximum F1 score")
print(best_f1)


Question 4: Maximum F1 score
threshold    0.410000
precision    0.619048
recall       0.976109
f1           0.757616
Name: 41, dtype: float64


## Question 5: 5-Fold Cross-Validation

In [7]:
kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=1
)

scores = []

for train_idx, val_idx in kfold.split(df_full_train):
    # Split the data
    df_fold_train = df_full_train.iloc[train_idx]
    df_fold_val = df_full_train.iloc[val_idx]

    # Separate target variables
    y_fold_train = df_fold_train["converted"].values
    y_fold_val = df_fold_val["converted"].values

    # Prepare features
    train_dicts = df_fold_train.drop(
        columns=["converted"]
    ).to_dict(orient="records")

    val_dicts = df_fold_val.drop(
        columns=["converted"]
    ).to_dict(orient="records")

    # One-hot encoding
    dv_fold = DictVectorizer(sparse=True)

    X_fold_train = dv_fold.fit_transform(train_dicts)
    X_fold_val = dv_fold.transform(val_dicts)

    # Train the model
    model_fold = LogisticRegression(
        solver="liblinear",
        C=1.0,
        max_iter=1000
    )

    model_fold.fit(X_fold_train, y_fold_train)

    # Predict probabilities
    y_fold_pred = model_fold.predict_proba(X_fold_val)[:, 1]

    # Calculate AUC
    auc = roc_auc_score(y_fold_val, y_fold_pred)
    scores.append(auc)

    print(f"Fold {len(scores)} AUC: {auc:.6f}")

print("\nMean AUC:", round(np.mean(scores), 3))
print("Standard deviation:", round(np.std(scores), 3))


Fold 1 AUC: 0.730560
Fold 2 AUC: 0.729573
Fold 3 AUC: 0.736343
Fold 4 AUC: 0.741245
Fold 5 AUC: 0.720914

Mean AUC: 0.732
Standard deviation: 0.007


## Question 6: Hyperparameter Tuning

In [8]:
C_values = [0.000001, 0.001, 1]

for C in C_values:
    kfold = KFold(
        n_splits=5,
        shuffle=True,
        random_state=1
    )

    scores = []

    for train_idx, val_idx in kfold.split(df_full_train):
        df_fold_train = df_full_train.iloc[train_idx]
        df_fold_val = df_full_train.iloc[val_idx]

        y_fold_train = df_fold_train["converted"].values
        y_fold_val = df_fold_val["converted"].values

        train_dicts = df_fold_train.drop(
            columns=["converted"]
        ).to_dict(orient="records")

        val_dicts = df_fold_val.drop(
            columns=["converted"]
        ).to_dict(orient="records")

        dv_fold = DictVectorizer(sparse=True)

        X_fold_train = dv_fold.fit_transform(train_dicts)
        X_fold_val = dv_fold.transform(val_dicts)

        model_fold = LogisticRegression(
            solver="liblinear",
            C=C,
            max_iter=1000
        )

        model_fold.fit(X_fold_train, y_fold_train)

        y_fold_pred = model_fold.predict_proba(X_fold_val)[:, 1]

        auc = roc_auc_score(y_fold_val, y_fold_pred)
        scores.append(auc)

    print(
        f"C={C}: mean AUC={np.mean(scores):.3f}, "
        f"std={np.std(scores):.3f}"
    )

C=1e-06: mean AUC=0.617, std=0.019
C=0.001: mean AUC=0.749, std=0.010
C=1: mean AUC=0.732, std=0.007
